In [ ]:
# LBKT Training Pipeline

import torch
import os
print('CUDA available:', torch.cuda.is_available())
print('PyTorch version:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Check if unified splits exist
splits_path = os.path.join('..', 'data', 'splits.json')
print(f'\nUnified splits exist: {os.path.exists(splits_path)}')

CUDA available: True
PyTorch version: 2.2.2+cu121
GPU: NVIDIA H200 MIG 2g.35gb

Unified splits exist: True


In [2]:
# Step 0: Generate unified splits (run this once, shared across all models)
# This ensures DKT, GKT, reKT, and LBKT all use the same train/valid/test split
# Uses stratified splitting to ensure all completion rate bins are represented

import os
splits_path = os.path.join('..', 'data', 'splits.json')

if not os.path.exists(splits_path):
    print("Generating unified splits...")
    !cd .. && python generate_splits.py \
        --input merged_student_question_history.csv \
        --output ../data/splits.json \
        --seed 42 \
        --stratify
else:
    print(f"Unified splits already exist at {splits_path}")

Unified splits already exist at ../data/splits.json


In [3]:
# Step 1: Preprocess the merged CSV data
# This uses the unified splits from data/splits.json (generated by generate_splits.py)
# Make sure to run generate_splits.py first if splits.json doesn't exist

!python lbkt_preprocess_merged.py \
  --input ../merged_student_question_history.csv \
  --output_dir lbkt_processed \
  --min_seq_len 3

Loading data from: ../merged_student_question_history.csv
Loaded 1590 rows, 5265 columns
Detected 305 question slots: q1_id ... q305_id
Extracting sequences...
Users with >= 3 interactions: 1590
Unique questions: 95
Loading unified splits from /uufs/chpc.utah.edu/common/home/u1389859/models/lbkt/../data/splits.json ...
Applied unified split: Train=1107, Valid=157, Test=326
Final split: train=1107, valid=157, test=326

Saved to lbkt_processed/
  - lbkt_train.pkl (1107 users)
  - lbkt_valid.pkl (157 users)
  - lbkt_test.pkl (326 users)
  - metadata.json
  - qid_to_idx.pkl
  - completion_rates.pkl

Sequence statistics:
  Min length: 34
  Max length: 305
  Mean length: 87.7
  Median length: 35.0


In [4]:
# Step 2: Train LBKT Model
# ========================
# LBKT: BERT-style Architecture + Rasch Embeddings + LSTM
# 
# Based on: "Integrating LSTM and BERT for Long-Sequence Data Analysis in ITS" (Li et al., 2024)
#
# Key components:
#   - BERT-style Transformer blocks (trained from scratch, NOT pre-trained)
#   - Rasch Embeddings: E_rasch = token + segment * (token + segment)
#   - LSTM for sequential knowledge state tracking
#
# Hyperparameters aligned with DKT/GKT/reKT for fair comparison

!python train_lbkt_merged.py \
  --data_dir lbkt_processed \
  --max_len 310 \
  --embed_dim 64 \
  --num_heads 4 \
  --num_layers 2 \
  --lstm_hidden 64 \
  --lstm_layers 1 \
  --dropout 0.2 \
  --batch_size 64 \
  --epochs 50 \
  --lr 0.001 \
  --warmup_epochs 5 \
  --device cuda \
  --model_type lbkt \
  --save_best lbkt_best.pt

Using device: cuda
Metadata: num_questions=95, total_q_slots=305.0
Train: 1107 users, Valid: 157 users, Test: 326 users
Model: lbkt
Parameters: 151,681

Epoch 1/50
  [Train] loss=0.6794, AUC=0.4954, ACC=0.5898, F1=0.7128
  Fairness per completion-rate bin:
    Bin 1 (10- 20%): students=82, predictions=2752, TPR=0.998, FPR=0.997, ACC=0.601, F1=0.750, AUC=0.509
    Bin 2 (20- 30%): students=2, predictions=137, TPR=1.000, FPR=1.000, ACC=0.737, F1=0.849, AUC=0.519
    Bin 3 (30- 40%): students=3, predictions=297, TPR=0.991, FPR=1.000, ACC=0.710, F1=0.831, AUC=0.552
    Bin 4 (40- 50%): students=49, predictions=6485, TPR=0.996, FPR=0.994, ACC=0.764, F1=0.866, AUC=0.518
    Bin 5 (50- 60%): students=12, predictions=2008, TPR=0.995, FPR=0.989, ACC=0.735, F1=0.847, AUC=0.502
    Bin 6 (60- 70%): students=7, predictions=1415, TPR=0.995, FPR=0.997, ACC=0.743, F1=0.852, AUC=0.497
    Bin 7 (70- 80%): students=1, predictions=238, TPR=0.994, FPR=1.000, ACC=0.655, F1=0.792, AUC=0.456
    Bin 8 (80- 

In [ ]:
# Optional: Load and inspect the best model

import torch
import json

# Load best model checkpoint
checkpoint = torch.load('lbkt_best.pt', weights_only=False)
print("Best model info:")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Valid AUC: {checkpoint['valid_auc']:.4f}")
print(f"  Hyperparameters:")
for k, v in checkpoint['args'].items():
    print(f"    {k}: {v}")

# Load metadata
with open('lbkt_processed/metadata.json', 'r') as f:
    metadata = json.load(f)
    print(f"\nDataset info:")
    print(f"  Number of questions: {metadata['num_questions']}")
    print(f"  Train students: {metadata['n_train']}")
    print(f"  Valid students: {metadata['n_valid']}")
    print(f"  Test students: {metadata['n_test']}")

In [ ]:
# Inspect model architecture

from model import LBKT

# Create model with same config as training
model = LBKT(
    num_questions=95,  # Update with actual number from metadata
    embed_dim=128,
    num_heads=4,
    num_layers=2,
    lstm_hidden=128,
    lstm_layers=1,
    max_seq_len=100,
    dropout=0.1,
)

print("LBKT Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## Alternative Configurations

### Quick Test Run (faster, for debugging)
```bash
python train_lbkt_merged.py \
  --data_dir lbkt_processed \
  --max_len 50 \
  --embed_dim 64 \
  --num_heads 2 \
  --num_layers 1 \
  --lstm_hidden 64 \
  --epochs 5 \
  --batch_size 32 \
  --device cuda
```

### LBKT Simple (Lightweight, no Transformer)
```bash
python train_lbkt_merged.py \
  --data_dir lbkt_processed \
  --max_len 100 \
  --embed_dim 128 \
  --lstm_hidden 128 \
  --lstm_layers 2 \
  --dropout 0.1 \
  --batch_size 64 \
  --epochs 30 \
  --model_type lbkt_simple \
  --device cuda
```

### High-Capacity Model (for larger datasets)
```bash
python train_lbkt_merged.py \
  --data_dir lbkt_processed \
  --max_len 200 \
  --embed_dim 256 \
  --num_heads 8 \
  --num_layers 4 \
  --lstm_hidden 256 \
  --lstm_layers 2 \
  --dropout 0.1 \
  --batch_size 32 \
  --epochs 50 \
  --lr 5e-4 \
  --device cuda
```

### CPU Training (no GPU)
```bash
python train_lbkt_merged.py \
  --data_dir lbkt_processed \
  --max_len 100 \
  --embed_dim 64 \
  --num_heads 2 \
  --num_layers 1 \
  --lstm_hidden 64 \
  --batch_size 32 \
  --epochs 30 \
  --device cpu
```

## Model Details

### LBKT Architecture

Based on: "Integrating LSTM and BERT for Long-Sequence Data Analysis in ITS" (Li et al., 2024)

**Key insight:** The paper uses a BERT-style **architecture** (transformer blocks with attention), but trains from scratch - it does NOT use pre-trained BERT weights.

### Model Variants

| Variant | Model Type | Description |
|---------|------------|-------------|
| **LBKT** (Full) | `lbkt` | Transformer blocks + Rasch embeddings + LSTM |
| **LBKT Simple** | `lbkt_simple` | Rasch embeddings + LSTM (no transformer) |

### LBKT Components (`--model_type lbkt`)

1. **Rasch Embedding**: `E_rasch = E_token + E_segment * (E_token + E_segment)`
   - `E_token`: Encodes question + response (q_id + correct * n_skill)
   - `E_segment`: Question ID embedding (acts as difficulty modifier)

2. **Transformer Blocks**: Multi-head self-attention + Feed-forward
   - Follows BERT architecture pattern
   - Trained from scratch (NOT pre-trained)

3. **LSTM**: Sequential knowledge state tracking
   - Key for handling long sequences (>400 interactions)

### Reference
Li, S., Song, J., & McTavish, T. (2024). Integrating LSTM and BERT for Long-Sequence Data Analysis in Intelligent Tutoring Systems.